# Fast opening with rustytree

The ISMIP7 forcing datasets on [source.coop](https://source.coop/ismip) are ~285,000
NetCDF files. The [englacial/ismip7](https://source.coop/englacial/ismip7) Icechunk
stores hold *virtual references* into those files: no data is copied, every group
(`<ESM>/<experiment>/<product>`) is one xarray Dataset with a normalised time axis,
exact standard-grid coordinates and a correct `crs`. Reading is anonymous.

Set `ISMIP7_STORE_AIS` / `ISMIP7_STORE_GRIS` to `local:/path/to/repo` to run these
notebooks against a local dev store.


In [1]:
import os
import time

from ismip7_helper.access import list_groups, store_target


def read(a, tries=3):
    """Load a lazy array and return the seconds taken; retry on transient object-store errors."""
    for attempt in range(tries):
        t0 = time.time()
        try:
            a.load()
            return round(time.time() - t0, 2)
        except Exception as e:
            if attempt == tries - 1:
                raise
            print("retry after", type(e).__name__)


REGION = os.environ.get("ISMIP7_NB_REGION", "AIS")
print("store:", store_target(REGION).url)
groups = list_groups(REGION)
print(len(groups), "groups; first:", groups[:5])
GROUP = os.environ.get("ISMIP7_NB_GROUP", groups[0])

store: local:/tmp/claude-0/-home-user-ismip7-indexing/c378e836-2097-52d9-9f11-06a02384e3a8/scratchpad/devstore/icechunk-ais


  2026-11-01T00:57:29.549523Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



2 groups; first: ['CESM2-WACCM/historical/SDBN1-8000m', 'CESM2-WACCM/historical/ocean']


[rustytree](https://github.com/aladinor/rustytree) walks the Zarr v3 hierarchy in Rust, concurrently, and plugs into xarray as `engine='rustytree'`. The ISMIP7 stores are Icechunk spec v2 with time-split manifests, which it reads directly.

In [2]:
from ismip7_helper.access import open_store

timings = {}
for engine in ("zarr", "rustytree"):
    for run in range(2):
        t0 = time.time()
        tree = open_store(REGION, engine=engine)
        dt_open = time.time() - t0
        leaf = next(g for g in tree.subtree if g.data_vars)
        var = next(iter(leaf.data_vars))
        dt_read = read(leaf[var].isel(time=0))
        timings[(engine, run)] = (dt_open, dt_read)
        print(
            f"{engine:9s} run{run}: open {dt_open:.2f}s, first timestep {dt_read:.2f}s ({leaf.path}/{var})"
        )

  2026-11-01T00:57:30.824775Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



zarr      run0: open 0.07s, first timestep 0.63s (/CESM2-WACCM/historical/SDBN1-8000m/acabf)


  2026-11-01T00:57:31.522299Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



zarr      run1: open 0.06s, first timestep 0.64s (/CESM2-WACCM/historical/SDBN1-8000m/acabf)


  2026-11-01T00:57:32.215944Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



rustytree run0: open 0.03s, first timestep 0.66s (/CESM2-WACCM/historical/SDBN1-8000m/acabf)


  2026-11-01T00:57:32.904587Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



rustytree run1: open 0.01s, first timestep 0.64s (/CESM2-WACCM/historical/SDBN1-8000m/acabf)


## Filter groups while opening

`group_filter` (rustytree only) returns a tree with just the matching groups, e.g. one ESM's scenarios.

In [3]:
pattern = "/" + GROUP.split("/")[0] + "/*/*"
sub = open_store(REGION, engine="rustytree", group_filter=pattern)
print(pattern, "->", [g.path for g in sub.subtree if g.data_vars][:10])

/CESM2-WACCM/*/* -> ['/CESM2-WACCM/historical/SDBN1-8000m', '/CESM2-WACCM/historical/ocean']


  2026-11-01T00:57:33.562804Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



In [4]:
from ismip7_helper.access import open_group

ds = open_group(REGION, GROUP, engine="rustytree")
ds

  2026-11-01T00:57:33.582765Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



<xarray.Dataset> Size: 167MB
Dimensions:  (time: 36, y: 761, x: 761)
Coordinates:
  * time     (time) datetime64[ns] 288B 1850-01-15 1850-02-15 ... 1852-12-15
  * y        (y) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
  * x        (x) float64 6kB -3.04e+06 -3.032e+06 ... 3.032e+06 3.04e+06
    crs      int32 4B ...
Data variables:
    acabf    (time, y, x) float32 83MB ...
    tas      (time, y, x) float32 83MB ...
Attributes: (12/21)
    Conventions:            CF-1.6
    comment:                Prepared for ISMIP7 by Devon Dunmire using xesmf ...
    grid:                   Map Projection: Polar Stereographic South EPSG303...
    grid_mapping:           crs
    institution:            ULiege (Brice Noel)
    ismip7_dataset:         ismip7-ais-forcing
    ...                     ...
    regrid_method:          conservative
    source:                 SDBN1 - Brice Noel -  bnoel@uliege.be
    source_keys_prefix:     ismip/ismip7-ais-forcing/data/CESM2-WACCM/histori...
    source_time_calendar:   standard
    source_time_units:      days since 1850-01-01 00:00:00
    spatial_resolution:     8000 m